In [1]:
import pandas as pd
import glob
import os

print("INICIANDO PIPELINE: RAW -> BRONZE (ONS)\n" + "-"*50)

# ==============================================================================
# 1. PROCESSAMENTO DO VOLUME ÚTIL DIÁRIO
# ==============================================================================
print("PARTE 1: Extraindo dados de Volume da Represa...")
caminho_raw_vol = "../data/raw/ons/daily/"
caminho_bronze_dir = "../data/processed/ons/"
os.makedirs(caminho_bronze_dir, exist_ok=True)

arquivos_vol = glob.glob(os.path.join(caminho_raw_vol, "*.csv"))
lista_furnas_vol = []

for arquivo in arquivos_vol:
    ano_arquivo = os.path.basename(arquivo)
    print(f"  -> Lendo e filtrando Volume {ano_arquivo}...")
    try:
        df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8")
    except UnicodeDecodeError:
        df_temp = pd.read_csv(arquivo, sep=";", encoding="latin1")
        
    df_filtrado = df_temp[df_temp["nom_reservatorio"].str.contains("FURNAS", case=False, na=False)].copy()
    lista_furnas_vol.append(df_filtrado)

if lista_furnas_vol:
    df_bronze_vol = pd.concat(lista_furnas_vol, ignore_index=True)
    caminho_bronze_vol = os.path.join(caminho_bronze_dir, "furnas_ons_bronze.csv")
    df_bronze_vol.to_csv(caminho_bronze_vol, index=False)
    print(f"SUCESSO! Volume salvo: {caminho_bronze_vol} ({len(df_bronze_vol)} registros)\n")
else:
    print("Nenhum arquivo de Volume encontrado na pasta raw/ons/daily/\n")


# ==============================================================================
# 2. PROCESSAMENTO DA ENERGIA NATURAL AFLUENTE (ENA / CHUVA)
# ==============================================================================
print("PARTE 2: Extraindo dados de Clima/Vazão (ENA)...")
caminho_raw_ena = "../data/raw/ons/ena/"

arquivos_ena = glob.glob(os.path.join(caminho_raw_ena, "*.csv"))
lista_furnas_ena = []

for arquivo in arquivos_ena:
    ano_arquivo = os.path.basename(arquivo)
    print(f"  -> Lendo e filtrando ENA {ano_arquivo}...")
    try:
        df_temp = pd.read_csv(arquivo, sep=";", encoding="utf-8")
    except UnicodeDecodeError:
        df_temp = pd.read_csv(arquivo, sep=";", encoding="latin1")
        
    df_filtrado = df_temp[df_temp["nom_reservatorio"].str.contains("FURNAS", case=False, na=False)].copy()
    lista_furnas_ena.append(df_filtrado)

if lista_furnas_ena:
    df_bronze_ena = pd.concat(lista_furnas_ena, ignore_index=True)
    caminho_bronze_ena = os.path.join(caminho_bronze_dir, "furnas_ena_bronze.csv")
    df_bronze_ena.to_csv(caminho_bronze_ena, index=False)
    print(f"SUCESSO! ENA salva: {caminho_bronze_ena} ({len(df_bronze_ena)} registros)\n")
else:
    print("Nenhum arquivo de ENA encontrado na pasta raw/ons/ena/\n")

print("-" * 50)
print("PIPELINE FINALIZADO!")

# ==============================================================================
# 3. PROCESSAMENTO DA GERAÇÃO TÉRMICA (EIXO 3)
# ==============================================================================
print("PARTE 3: Extraindo dados de Geração Térmica...")
caminho_raw_termicas = "../data/raw/ons/termicas/"

arquivos_termicas = glob.glob(os.path.join(caminho_raw_termicas, "*.csv"))
lista_termicas = []

def limpa_numero(x):
    if pd.isna(x): return 0.0
    try:
        return float(str(x).replace(',', '.'))
    except:
        return 0.0

for arquivo in arquivos_termicas:
    ano_arquivo = os.path.basename(arquivo)
    print(f"  -> Lendo e limpando Geração {ano_arquivo}...")
    
    try:
        df_temp = pd.read_csv(arquivo, sep=";", dtype=str, encoding="utf-8")
    except UnicodeDecodeError:
        df_temp = pd.read_csv(arquivo, sep=";", dtype=str, encoding="latin1")
        
    colunas_interesse = [
        'din_instante', 'val_verifgeracao', 'val_verifgarantiaenergetica', 'val_verifordemmerito'
    ]
    
    cols_presentes = [col for col in colunas_interesse if col in df_temp.columns]
    df_filtrado = df_temp[cols_presentes].copy()
    
    if 'din_instante' in df_filtrado.columns:
        df_filtrado['din_instante'] = pd.to_datetime(df_filtrado['din_instante'], format='mixed', dayfirst=False).dt.date
        
    for col in ['val_verifgeracao', 'val_verifgarantiaenergetica', 'val_verifordemmerito']:
        if col in df_filtrado.columns:
            df_filtrado[col] = df_filtrado[col].apply(limpa_numero)
    
    lista_termicas.append(df_filtrado)

if lista_termicas:
    df_bronze_termicas = pd.concat(lista_termicas, ignore_index=True)
    df_nacional = df_bronze_termicas.groupby('din_instante').sum().reset_index()
    
    caminho_bronze_termicas = os.path.join(caminho_bronze_dir, "termicas_ons_bronze.csv")
    df_nacional.to_csv(caminho_bronze_termicas, index=False)
    print(f"SUCESSO! Térmicas salvas: {caminho_bronze_termicas} ({len(df_nacional)} registros)\n")
else:
    print("Nenhum arquivo de Térmicas encontrado na pasta raw/ons/termicas/\n")

print("-" * 50)
print("PIPELINE FINALIZADO!")

INICIANDO PIPELINE: RAW -> BRONZE (ONS)
--------------------------------------------------
PARTE 1: Extraindo dados de Volume da Represa...
  -> Lendo e filtrando Volume 2020.csv...
  -> Lendo e filtrando Volume 2021.csv...
  -> Lendo e filtrando Volume 2019.csv...
  -> Lendo e filtrando Volume 2025.csv...
  -> Lendo e filtrando Volume 2018.csv...
  -> Lendo e filtrando Volume 2026.csv...
  -> Lendo e filtrando Volume 2024.csv...
  -> Lendo e filtrando Volume 2022.csv...
  -> Lendo e filtrando Volume 2023.csv...
  -> Lendo e filtrando Volume 2017.csv...
SUCESSO! Volume salvo: ../data/processed/ons/furnas_ons_bronze.csv (3369 registros)

PARTE 2: Extraindo dados de Clima/Vazão (ENA)...
  -> Lendo e filtrando ENA 2020.csv...
  -> Lendo e filtrando ENA 2021.csv...
  -> Lendo e filtrando ENA 2019.csv...
  -> Lendo e filtrando ENA 2025.csv...
  -> Lendo e filtrando ENA 2018.csv...
  -> Lendo e filtrando ENA 2026.csv...
  -> Lendo e filtrando ENA 2024.csv...
  -> Lendo e filtrando ENA 2022.c